In [39]:
import ee
import json
import math
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import base64
from shapely.geometry import Point
from scipy.spatial.distance import cdist
from scipy.optimize import linear_sum_assignment
from PIL import Image
from io import BytesIO

# Initialize Earth Engine
try:
    ee.Initialize(project="gsapp-map")
except Exception:
    ee.Authenticate()
    ee.Initialize(project="gsapp-map")

In [40]:
# Configuration
SQUARE_VALUES = [64, 144, 256, 576, 900, 1024]  # Perfect squares for 2D (8², 12², 16², 24², 30²)
CUBE_VALUES = [25, 64, 125, 216, 512, 1000]    # Perfect cubes for 3D (4³, 5³, 6³, 8³, 10³)
N_VALUES = sorted(set(SQUARE_VALUES + CUBE_VALUES))  # Union: precompute all unique values

EMBED_PIXELS = 64
EMBED_SCALE = 10
YEAR = 2024

GEOJSON_PATH = "../dimension-reduction/data/2000_sampled_classified_embeddings.geojson"
OUTPUT_DIR = "data"
PATCH_DIR = "sat_patches"

os.makedirs(PATCH_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Will precompute for N values: {N_VALUES}")
print(f"  2D (squares): {SQUARE_VALUES}")
print(f"  3D (cubes): {CUBE_VALUES}")

Will precompute for N values: [25, 64, 125, 144, 216, 256, 512, 576, 900, 1000, 1024]
  2D (squares): [64, 144, 256, 576, 900, 1024]
  3D (cubes): [25, 64, 125, 216, 512, 1000]


In [44]:
# Load GeoJSON
print(f"Loading GeoJSON from {GEOJSON_PATH} ...")

with open(GEOJSON_PATH, "r") as f:
    geojson_data = json.load(f)

# Take maximum N needed (1000)
features = geojson_data["features"][:max(N_VALUES)]
data = []

for i, feat in enumerate(features):
    props = feat["properties"]
    coords = feat["geometry"]["coordinates"]
    
    data.append({
        "index": i,
        "lon": coords[0],
        "lat": coords[1],
        "classification": props.get("classification"),
        "subregion_name": props.get("subregion_name", "Unknown"),
        "umap_1d_x": props.get("umap_1d_x"),
        "umap_2d_x": props.get("umap_2d_x"),
        "umap_2d_y": props.get("umap_2d_y"),
        "umap_3d_x": props.get("umap_3d_x"),
        "umap_3d_y": props.get("umap_3d_y"),
        "umap_3d_z": props.get("umap_3d_z"),
        "tsne_1d_x": props.get("tsne_1d_x"),
        "tsne_2d_x": props.get("tsne_2d_x"),
        "tsne_2d_y": props.get("tsne_2d_y"),
        "tsne_3d_x": props.get("tsne_3d_x"),
        "tsne_3d_y": props.get("tsne_3d_y"),
        "tsne_3d_z": props.get("tsne_3d_z"),
    })

gdf = gpd.GeoDataFrame(
    data, geometry=[Point(d["lon"], d["lat"]) for d in data], crs="EPSG:4326"
)

print(f"Loaded {len(gdf)} samples")

Loading GeoJSON from ../dimension-reduction/data/2000_sampled_classified_embeddings.geojson ...
Loaded 1024 samples


In [45]:
gdf.head()

,index,lon,lat,classification,subregion_name,umap_1d_x,umap_2d_x,umap_2d_y,umap_3d_x,umap_3d_y,umap_3d_z,tsne_1d_x,tsne_2d_x,tsne_2d_y,tsne_3d_x,tsne_3d_y,tsne_3d_z,geometry
0,0,-131.184025,59.611055,126,Northern America,16.784231,-40.799011,-2.798702,-45.941341,4.641312,-26.571562,-49.619076,-26.663736,13.954807,-16.747684,38.547501,29.680862,POINT (-131.18402 59.61106)
1,1,-62.623884,9.017345,30,Latin America and the Caribbean,0.639102,10.030628,43.123466,6.111301,49.059601,24.983414,17.551109,19.866989,-37.949833,36.957924,-40.670971,20.553169,POINT (-62.62388 9.01735)
2,2,11.678798,-3.611274,30,Sub-Saharan Africa,-12.320284,48.158806,-12.502239,36.104492,-3.900765,27.594849,49.514023,40.090767,-2.841045,-2.776424,-47.970524,-25.487997,POINT (11.6788 -3.61127)
3,3,10.361996,45.732221,115,Southern Europe,13.234972,-24.740774,-2.040253,-35.001968,-5.382343,-0.889820,-24.178932,-10.349129,7.788277,-24.244860,5.883209,0.029480,POINT (10.362 45.73222)
4,4,32.748271,58.383271,126,Eastern Europe,21.671848,-27.601662,-13.733841,-35.775017,-11.860596,-15.688488,-22.066343,-14.942474,15.959877,-30.494827,11.660652,16.022049,POINT (32.74827 58.38327)


In [35]:
# Helper function to arrange points in grid using Hungarian algorithm
def arrange_to_grid_1d(values, n_points):
    """Arrange 1D values to 1D grid positions (0 to n_points-1)"""
    grid_pos = np.arange(n_points).reshape(-1, 1)
    values_2d = values.reshape(-1, 1)
    
    cost = cdist(values_2d, grid_pos)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros(n_points)
    result[r_idx] = grid_pos[c_idx].flatten()
    return result


def arrange_to_grid_2d(coords_2d, n_points):
    """Arrange 2D coords to square grid"""
    grid_side = int(math.ceil(math.sqrt(n_points)))
    grid_x = np.linspace(0, grid_side - 1, grid_side)
    grid_y = np.linspace(0, grid_side - 1, grid_side)
    grid_coords = np.array(np.meshgrid(grid_x, grid_y)).T.reshape(-1, 2)[:n_points]
    
    cost = cdist(coords_2d, grid_coords)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros((n_points, 2))
    result[r_idx] = grid_coords[c_idx]
    return result


def arrange_to_grid_3d(coords_3d, n_points):
    """Arrange 3D coords to cube grid"""
    cube_side = int(math.ceil(n_points ** (1/3)))
    grid_x = np.linspace(0, cube_side - 1, cube_side)
    grid_y = np.linspace(0, cube_side - 1, cube_side)
    grid_z = np.linspace(0, cube_side - 1, cube_side)
    grid_coords = np.array(np.meshgrid(grid_x, grid_y, grid_z)).T.reshape(-1, 3)[:n_points]
    
    cost = cdist(coords_3d, grid_coords)
    r_idx, c_idx = linear_sum_assignment(cost)
    
    result = np.zeros((n_points, 3))
    result[r_idx] = grid_coords[c_idx]
    return result

print("Grid arrangement functions defined")

Grid arrangement functions defined


In [46]:
# Generate grid positions for each N value
all_gdfs = {}

for n in N_VALUES:
    print(f"\n=== Processing N = {n} ===")
    
    # Get subset of data
    gdf_subset = gdf.iloc[:n].copy()
    
    # UMAP 1D
    print(f"Arranging UMAP 1D for {n} samples...")
    umap_1d = gdf_subset["umap_1d_x"].values.astype(np.float32)
    grid_1d = arrange_to_grid_1d(umap_1d, n)
    gdf_subset[f"grid_umap_1d_x"] = grid_1d
    
    # UMAP 2D
    print(f"Arranging UMAP 2D for {n} samples...")
    umap_2d = gdf_subset[["umap_2d_x", "umap_2d_y"]].values.astype(np.float32)
    grid_2d = arrange_to_grid_2d(umap_2d, n)
    gdf_subset[f"grid_umap_2d_x"] = grid_2d[:, 0]
    gdf_subset[f"grid_umap_2d_y"] = grid_2d[:, 1]
    
    # UMAP 3D
    print(f"Arranging UMAP 3D for {n} samples...")
    umap_3d = gdf_subset[["umap_3d_x", "umap_3d_y", "umap_3d_z"]].values.astype(np.float32)
    grid_3d = arrange_to_grid_3d(umap_3d, n)
    gdf_subset[f"grid_umap_3d_x"] = grid_3d[:, 0]
    gdf_subset[f"grid_umap_3d_y"] = grid_3d[:, 1]
    gdf_subset[f"grid_umap_3d_z"] = grid_3d[:, 2]
    
    # t-SNE 1D
    print(f"Arranging t-SNE 1D for {n} samples...")
    tsne_1d = gdf_subset["tsne_1d_x"].values.astype(np.float32)
    grid_1d = arrange_to_grid_1d(tsne_1d, n)
    gdf_subset[f"grid_tsne_1d_x"] = grid_1d
    
    # t-SNE 2D
    print(f"Arranging t-SNE 2D for {n} samples...")
    tsne_2d = gdf_subset[["tsne_2d_x", "tsne_2d_y"]].values.astype(np.float32)
    grid_2d = arrange_to_grid_2d(tsne_2d, n)
    gdf_subset[f"grid_tsne_2d_x"] = grid_2d[:, 0]
    gdf_subset[f"grid_tsne_2d_y"] = grid_2d[:, 1]
    
    # t-SNE 3D
    print(f"Arranging t-SNE 3D for {n} samples...")
    tsne_3d = gdf_subset[["tsne_3d_x", "tsne_3d_y", "tsne_3d_z"]].values.astype(np.float32)
    grid_3d = arrange_to_grid_3d(tsne_3d, n)
    gdf_subset[f"grid_tsne_3d_x"] = grid_3d[:, 0]
    gdf_subset[f"grid_tsne_3d_y"] = grid_3d[:, 1]
    gdf_subset[f"grid_tsne_3d_z"] = grid_3d[:, 2]
    
    all_gdfs[n] = gdf_subset

print("\nGrid positions generated for all N values")


=== Processing N = 25 ===
Arranging UMAP 1D for 25 samples...
Arranging UMAP 2D for 25 samples...
Arranging UMAP 3D for 25 samples...
Arranging t-SNE 1D for 25 samples...
Arranging t-SNE 2D for 25 samples...
Arranging t-SNE 3D for 25 samples...

=== Processing N = 64 ===
Arranging UMAP 1D for 64 samples...
Arranging UMAP 2D for 64 samples...
Arranging UMAP 3D for 64 samples...
Arranging t-SNE 1D for 64 samples...
Arranging t-SNE 2D for 64 samples...
Arranging t-SNE 3D for 64 samples...

=== Processing N = 125 ===
Arranging UMAP 1D for 125 samples...
Arranging UMAP 2D for 125 samples...
Arranging UMAP 3D for 125 samples...
Arranging t-SNE 1D for 125 samples...
Arranging t-SNE 2D for 125 samples...
Arranging t-SNE 3D for 125 samples...

=== Processing N = 144 ===
Arranging UMAP 1D for 144 samples...
Arranging UMAP 2D for 144 samples...
Arranging UMAP 3D for 144 samples...
Arranging t-SNE 1D for 144 samples...
Arranging t-SNE 2D for 144 samples...
Arranging t-SNE 3D for 144 samples...

=

In [48]:
# Download satellite patches (only need max N)
max_n = max(N_VALUES)
print(f"\nDownloading satellite patches for {max_n} locations...")

# Two datasets: embedding and true color satellite
embedding_dataset = ee.ImageCollection("GOOGLE/SATELLITE_EMBEDDING/V1/ANNUAL")
satellite_dataset = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
landsat_dataset = ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")

for idx, row in gdf.iterrows():
    patch_embed_path = os.path.join(PATCH_DIR, f"patch_{idx:04d}_embed.png")
    patch_rgb_path = os.path.join(PATCH_DIR, f"patch_{idx:04d}_rgb.png")
    
    point = ee.Geometry.Point(row.lon, row.lat)
    region = point.buffer(EMBED_SCALE * EMBED_PIXELS / 2).bounds()
    
    # Download embedding image
    if not os.path.exists(patch_embed_path):
        try:
            # Get embedding image
            embed_img = (
                embedding_dataset.filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
                .filterBounds(region)
                .mosaic()
                .select(["A01", "A02", "A03"])  # Embedding RGB bands
            )
            
            # Download as thumbnail
            url = embed_img.getThumbURL({
                'region': region,
                'dimensions': f'{EMBED_PIXELS}x{EMBED_PIXELS}',
                'format': 'png',
                'min': -0.3,
                'max': 0.3
            })
            
            # Download and save
            import requests
            response = requests.get(url)
            if response.status_code == 200:
                with open(patch_embed_path, 'wb') as f:
                    f.write(response.content)
        
        except Exception as e:
            print(f"  Error downloading embedding at {idx}: {e}")
    
    # Download true color satellite image
    if not os.path.exists(patch_rgb_path):
        rgb_img = None
        
        # Strategy 1: Try Sentinel-2 with cloud filter
        try:
            rgb_img = (
                satellite_dataset
                .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
                .filterBounds(region)
                .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))
                .median()
            )
            # Test if image has bands
            band_names = rgb_img.bandNames().getInfo()
            if len(band_names) > 0:
                rgb_img = rgb_img.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000)
            else:
                rgb_img = None
        except Exception as e:
            rgb_img = None
        
        # Strategy 2: Try Sentinel-2 without cloud filter
        if rgb_img is None:
            try:
                rgb_img = (
                    satellite_dataset
                    .filterDate(f"{YEAR}-01-01", f"{YEAR + 1}-01-01")
                    .filterBounds(region)
                    .median()
                )
                band_names = rgb_img.bandNames().getInfo()
                if len(band_names) > 0:
                    rgb_img = rgb_img.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000)
                else:
                    rgb_img = None
            except Exception as e:
                rgb_img = None
        
        # Strategy 3: Try Sentinel-2 from multiple years
        if rgb_img is None:
            try:
                rgb_img = (
                    satellite_dataset
                    .filterDate(f"{YEAR-2}-01-01", f"{YEAR + 1}-01-01")
                    .filterBounds(region)
                    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
                    .median()
                )
                band_names = rgb_img.bandNames().getInfo()
                if len(band_names) > 0:
                    rgb_img = rgb_img.select(['B4', 'B3', 'B2']).visualize(min=0, max=3000)
                else:
                    rgb_img = None
            except Exception as e:
                rgb_img = None
        
        # Strategy 4: Try Landsat 8
        if rgb_img is None:
            try:
                def scale_landsat(image):
                    optical_bands = image.select('SR_B.').multiply(0.0000275).add(-0.2)
                    return image.addBands(optical_bands, None, True)
                
                rgb_img = (
                    landsat_dataset
                    .filterDate(f"{YEAR-2}-01-01", f"{YEAR + 1}-01-01")
                    .filterBounds(region)
                    .filter(ee.Filter.lt('CLOUD_COVER', 30))
                    .map(scale_landsat)
                    .median()
                )
                band_names = rgb_img.bandNames().getInfo()
                if len(band_names) > 0:
                    rgb_img = rgb_img.select(['SR_B4', 'SR_B3', 'SR_B2']).visualize(min=0, max=0.3)
                else:
                    rgb_img = None
            except Exception as e:
                rgb_img = None
        
        # Download if we got an image
        if rgb_img is not None:
            try:
                url = rgb_img.getThumbURL({
                    'region': region,
                    'dimensions': f'{EMBED_PIXELS}x{EMBED_PIXELS}',
                    'format': 'png'
                })
                
                response = requests.get(url)
                if response.status_code == 200:
                    with open(patch_rgb_path, 'wb') as f:
                        f.write(response.content)
            except Exception as e:
                print(f"  Error downloading RGB at {idx}: {e}")
        else:
            print(f"  No RGB imagery available at {idx}")
    
    if idx % 100 == 0:
        embed_status = "✓" if os.path.exists(patch_embed_path) else "✗"
        rgb_status = "✓" if os.path.exists(patch_rgb_path) else "✗"
        print(f"  {idx}/{max_n} - Embed: {embed_status} RGB: {rgb_status}")

print("\nSatellite patches complete")


  0/1024 - Embed: ✓ RGB: ✓
  100/1024 - Embed: ✓ RGB: ✓
  200/1024 - Embed: ✓ RGB: ✓
  300/1024 - Embed: ✓ RGB: ✓
  400/1024 - Embed: ✓ RGB: ✓
  500/1024 - Embed: ✓ RGB: ✓
  600/1024 - Embed: ✓ RGB: ✓
  700/1024 - Embed: ✓ RGB: ✓
  800/1024 - Embed: ✓ RGB: ✓
  900/1024 - Embed: ✓ RGB: ✓
  1000/1024 - Embed: ✓ RGB: ✓

Satellite patches complete


In [47]:
# Save GeoJSON files for each N value
print("\nSaving GeoJSON files...")

for n, gdf_subset in all_gdfs.items():
    # Add patch filenames to GeoDataFrame (both embedding and RGB)
    gdf_subset["patch_file_embed"] = [f"patch_{i:04d}_embed.png" for i in gdf_subset["index"]]
    gdf_subset["patch_file_rgb"] = [f"patch_{i:04d}_rgb.png" for i in gdf_subset["index"]]
    
    # Save to GeoJSON
    output_path = os.path.join(OUTPUT_DIR, f"web_grid_data_{n}.geojson")
    gdf_subset.to_file(output_path, driver="GeoJSON")
    print(f"  Saved {len(gdf_subset)} features to {output_path}")

print("\nAll files saved successfully!")
print(f"N values: {N_VALUES}")


Saving GeoJSON files...
  Saved 25 features to data/web_grid_data_25.geojson
  Saved 64 features to data/web_grid_data_64.geojson
  Saved 125 features to data/web_grid_data_125.geojson
  Saved 144 features to data/web_grid_data_144.geojson
  Saved 216 features to data/web_grid_data_216.geojson
  Saved 256 features to data/web_grid_data_256.geojson
  Saved 512 features to data/web_grid_data_512.geojson
  Saved 576 features to data/web_grid_data_576.geojson
  Saved 900 features to data/web_grid_data_900.geojson
  Saved 1000 features to data/web_grid_data_1000.geojson
  Saved 1024 features to data/web_grid_data_1024.geojson

All files saved successfully!
N values: [25, 64, 125, 144, 216, 256, 512, 576, 900, 1000, 1024]
